# Fitting the ESIS-I distortion

This notebook fits the alignment/distortion parameters of the ESIS-I optical
model to a Level-1 flight image, using a synthetic scene built from AIA
imagery captured during the flight.

It starts from the per-channel best fit recorded in
`esis.flights.f1.optics.distortion_fit()` and performs a **coupled polish** of
all four channels at once: parameters that correspond to a single physical
part (the field stop, the primary mirror, and the pointing of the payload)
are shared between the channels, while the per-grating parameters remain
independent.

Requirements:

* The `JSOC_EMAIL` environment variable must be set to an email address
  [registered with JSOC](http://jsoc.stanford.edu/ajax/register_email.html)
  (only needed the first time; the AIA scene is cached afterwards).
* Set `quick = False` below for a production fit (hours); the default
  settings run a minutes-long demonstration fit.

In [ ]:
import dataclasses
import pathlib
from datetime import datetime
import numpy as np
import matplotlib.pyplot as plt
import astropy.units as u
import named_arrays as na
import optika
import esis

In [ ]:
# fast demonstration settings; set False for a production fit
quick = True

num_scene = 201 if quick else 801

## Data

Load the Level-1 frame that the original distortion fit was optimized
against, and the AIA frame captured closest to it.

In [ ]:
time_index = 15

l1 = esis.flights.f1.data.level_1()[dict(time=time_index)]
time_esis = l1.inputs.time_start[dict(channel=0)].ndarray
time_esis

In [ ]:
scene = esis.flights.f1.data.synth.scene_aia()
scene = scene[np.argmin(np.abs((scene.inputs.time - time_esis).mean("wavelength")))]
scene.outputs.shape

Downsample the scene onto a regular grid to make the raytrace cheaper.
The scene radiance is regridded conservatively so that the total flux is
preserved.

In [ ]:
inputs_scene = na.TemporalSpectralPositionalVectorArray(
    time=scene.inputs.time,
    wavelength=scene.inputs.wavelength,
    position=na.Cartesian2dVectorArray(
        x=na.ScalarLinearSpace(
            scene.inputs.position.x.min(),
            scene.inputs.position.x.max(),
            axis="detector_x",
            num=num_scene,
        ),
        y=na.ScalarLinearSpace(
            scene.inputs.position.y.min(),
            scene.inputs.position.y.max(),
            axis="detector_y",
            num=num_scene,
        ),
    ),
)
scene_fit = scene(inputs_scene, axis=("detector_x", "detector_y"), method="conservative")
scene_fit.outputs.shape

In [ ]:
spectral_lines = ["O V", "Mg X", "He I"]

fig, ax = plt.subplots(ncols=3, figsize=(12, 4), constrained_layout=True)
for i in range(3):
    index = dict(wavelength=i, velocity=0)
    ax[i].imshow(scene_fit.outputs[index].ndarray.value.T, origin="lower")
    ax[i].set_title(spectral_lines[i]);

## Model

Start from the hard-coded per-channel best fit and idealize the materials,
so that the modeled counts are not modulated by the wavelength-dependent
coating and filter efficiencies (the correlation merit is insensitive to
per-channel scale factors anyway).

In [ ]:
model = esis.flights.f1.optics.distortion_fit(num_distribution=0)

model.grating.material = optika.materials.Mirror()
model.primary_mirror.material = optika.materials.Mirror()
model.filter.material = None
model.camera.sensor.material = optika.sensors.materials.IdealSensorMaterial()

# a single pupil cell is sufficient for distortion mapping
pupil = na.Cartesian2dVectorLinearSpace(
    start=-0.25,
    stop=0.25,
    axis=na.Cartesian2dVectorArray("pupil_x", "pupil_y"),
    num=2,
)

## Starting comparison

Render the model image and compare it against the Level-1 data.

In [ ]:
kwargs_image = dict(
    pupil=pupil,
    axis_wavelength="velocity",
    axis_field=("detector_x", "detector_y"),
    noise=False,
)


def render(instrument):
    image = instrument.system.image(scene_fit, **kwargs_image)
    axis_extra = tuple(set(image.outputs.shape) - set(na.shape(l1.outputs)))
    return image.outputs.sum(axis_extra)


image_start = render(model)
image_start.shape

In [ ]:
def compare(image_model, title):
    fig, ax = na.plt.subplots(
        figsize=(12, 10),
        constrained_layout=True,
        axis_rows="channel",
        nrows=l1.shape["channel"],
        axis_cols="column",
        ncols=2,
        sharex=True,
        sharey=True,
    )
    fig.suptitle(title)
    data = (l1.outputs - l1.outputs.mean()) / l1.outputs.std()
    na.plt.pcolormesh(
        l1.inputs.pixel.x,
        l1.inputs.pixel.y,
        C=data.value,
        ax=ax[dict(column=0)],
        vmax=np.percentile(data.value, 99),
    )
    image_model = (image_model - image_model.mean()) / image_model.std()
    na.plt.pcolormesh(
        l1.inputs.pixel.x,
        l1.inputs.pixel.y,
        C=image_model.value,
        ax=ax[dict(column=1)],
        vmax=np.percentile(image_model.value, 99),
    )
    na.plt.text(
        x=0.01,
        y=0.97,
        s=l1.channel,
        transform=na.plt.transAxes(ax[dict(column=0)]),
        ax=ax[dict(column=0)],
        ha="left",
        va="top",
        color="white",
    )
    ax.ndarray[0, 0].set_title("Level-1 data")
    ax.ndarray[0, 1].set_title("model")


compare(image_start, "starting model (hard-coded per-channel fit)")

## Coupled parameters

The hard-coded fit lets every parameter vary independently per channel, but
several of them describe a single physical part, so their per-channel scatter
is unphysical:

* `roll_field_stop` — there is only one field stop,
* `displacement_primary` — there is only one primary mirror,
* `pitch`, `yaw`, `roll` — the payload has a single pointing.

Replace those with their channel means to form the coupled starting point.
The per-grating parameters (`yaw_grating`, `pitch_grating`, `roll_grating`,
`spacing_rulings`) remain independent, since each channel has its own
grating. This reduces the fit from 36 to 21 degrees of freedom.

In [ ]:
parameters = esis.optics.DistortionParameters.from_instrument(model)

parameters.roll_field_stop = parameters.roll_field_stop.mean()
parameters.displacement_primary = parameters.displacement_primary.mean()
parameters.pitch = parameters.pitch.mean()
parameters.yaw = parameters.yaw.mean()
parameters.roll = parameters.roll.mean()

print("degrees of freedom:", na.pack(parameters).shape["pack"])
parameters

## Polish bounds

Since this is a polish around a known-good starting point, the bounds are
tight absolute windows around the start, sized to comfortably contain the
per-channel scatter of the original fit.

In [ ]:
delta = esis.optics.DistortionParameters(
    yaw_grating=2 * u.arcmin,
    pitch_grating=2 * u.arcmin,
    roll_grating=0.25 * u.deg,
    roll_field_stop=1 * u.deg,
    spacing_rulings=5e-4 * u.um,
    displacement_primary=3 * u.mm,
    pitch=15 * u.arcsec,
    yaw=15 * u.arcsec,
    roll=0.5 * u.deg,
)


def shift(parameters, delta, sign):
    return esis.optics.DistortionParameters(**{
        field.name: getattr(parameters, field.name) + sign * getattr(delta, field.name)
        for field in dataclasses.fields(parameters)
    })


bounds = (shift(parameters, delta, -1), shift(parameters, delta, +1))

## Coupled polish

Fit all four channels at once. With `axis_channel="channel"` the correlation
merit is computed independently per channel and averaged, so per-channel
brightness differences in the data do not suppress it.

Convergence is logged to a timestamped directory
(`convergence_data.csv`, `full_output.log`, `convergence_plot.png`).

In [ ]:
directory = pathlib.Path(
    f"distortion_polish_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
)

x0 = na.pack(parameters).ndarray
lower = na.pack(bounds[0]).ndarray
upper = na.pack(bounds[1]).ndarray

if quick:
    rng = np.random.default_rng(42)
    init = lower + (upper - lower) * rng.uniform(size=(5, x0.size))
    init[0] = x0
    kwargs_optimizer = dict(init=init, maxiter=2, tol=1e6, polish=False)
else:
    kwargs_optimizer = dict(
        popsize=10,
        maxiter=200,
        workers=-1,
        updating="deferred",
        polish=True,
    )

fitted = esis.optics.fit_distortion(
    instrument=model,
    scene=scene_fit,
    observation=l1.outputs.value,
    bounds=bounds,
    parameters=parameters,
    pupil=pupil,
    axis_wavelength="velocity",
    axis_field=("detector_x", "detector_y"),
    axis_channel="channel",
    directory=directory,
    kwargs_optimizer=kwargs_optimizer,
)

In [ ]:
fitted

## Result

In [ ]:
model_fitted = fitted.to_instrument(model)
image_fitted = render(model_fitted)

compare(image_fitted, "coupled polish")

In [ ]:
objective = esis.optics.DistortionObjective(
    instrument=model,
    parameters=parameters,
    scene=scene_fit,
    observation=l1.outputs.value,
    pupil=pupil,
    axis_wavelength="velocity",
    axis_field=("detector_x", "detector_y"),
    axis_channel="channel",
)

print("merit at coupled start:", objective(na.pack(parameters).ndarray))
print("merit after polish:    ", objective(na.pack(fitted).ndarray))